# Fine-tune DistilBERT для класифікації тексту

Transfer Learning, кастомізація LLM і RAG
**Мета:** показати повний мінімальний pipeline fine-tuning transformer-моделі для text classification.

Для демонстрації використаємо DistilBERT як encoder-модель і навчимо маленький класифікатор звернень до підтримки:

- `TECHNICAL` - технічна проблема;
- `BILLING` - оплата, рахунки, тариф;
- `DELIVERY` - доставка або статус замовлення.

## Встановлення бібліотек

In [ ]:
# !pip -q install "transformers[torch]>=4.40" "datasets>=2.18" "accelerate>=0.28"


## 1. Імпорти та налаштування

`RUN_TRAINING = False` залишено за замовчуванням, щоб notebook можна було швидко переглянути без довгого навчання. Для реальної демонстрації поставте `True` і запустіть notebook зверху вниз.


In [1]:
from pathlib import Path
import inspect
import random

import numpy as np
import pandas as pd


import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Для українських текстів беремо multilingual DistilBERT.
# Для швидшого англомовного демо можна замінити на: "distilbert/distilbert-base-uncased".
MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"
OUTPUT_DIR = Path("models/distilbert_support_router")

RUN_TRAINING = True
USE_TINY_SMOKE_MODEL = True

if USE_TINY_SMOKE_MODEL:
    # Тільки для перевірки, що pipeline технічно запускається. Не використовуйте для змістовної якості.
    MODEL_NAME = "hf-internal-testing/tiny-random-distilbert"

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")
print(f"Model checkpoint: {MODEL_NAME}")


/Users/dmitrostefan/Edu/PythonAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
Model checkpoint: hf-internal-testing/tiny-random-distilbert


## 2. Малий dataset для демонстрації

У реальному fine-tuning dataset має бути значно більшим, з чіткими правилами розмітки та окремим validation/test split. Тут ми створюємо малий локальний набір, щоб не залежати від зовнішнього dataset.


In [2]:
raw_examples = [
    # TECHNICAL
    ("TECHNICAL", "Додаток зависає після входу в акаунт."),
    ("TECHNICAL", "Не можу скинути пароль, посилання не відкривається."),
    ("TECHNICAL", "На сторінці профілю постійно з'являється помилка 500."),
    ("TECHNICAL", "Файл завантажується до 80 відсотків і потім процес зупиняється."),
    ("TECHNICAL", "Після оновлення не працює пошук у кабінеті."),
    ("TECHNICAL", "Сайт дуже повільно відкриває звіти."),
    ("TECHNICAL", "Не приходить SMS-код для підтвердження входу."),
    ("TECHNICAL", "Кнопка збереження не реагує на натискання."),
    # BILLING
    ("BILLING", "З картки списали кошти двічі за один тариф."),
    ("BILLING", "Потрібен рахунок для бухгалтерії за минулий місяць."),
    ("BILLING", "Хочу змінити тарифний план перед наступним списанням."),
    ("BILLING", "Оплата пройшла, але підписка не активувалася."),
    ("BILLING", "Як отримати акт виконаних робіт за оплату?"),
    ("BILLING", "Поверніть кошти, бо я випадково оплатив неправильний план."),
    ("BILLING", "Не можу додати нову корпоративну картку."),
    ("BILLING", "У квитанції неправильний податковий номер компанії."),
    # DELIVERY
    ("DELIVERY", "Коли буде доставлено моє замовлення?"),
    ("DELIVERY", "Трекінг показує, що посилка два дні не рухається."),
    ("DELIVERY", "Кур'єр не зателефонував перед доставкою."),
    ("DELIVERY", "Потрібно змінити адресу отримання замовлення."),
    ("DELIVERY", "Замовлення прийшло не в повній комплектації."),
    ("DELIVERY", "Хочу перенести доставку на інший день."),
    ("DELIVERY", "Статус замовлення не оновлюється після відправлення."),
    ("DELIVERY", "Посилка повернулася на склад без пояснення."),
]

label_names = ["TECHNICAL", "BILLING", "DELIVERY"]
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}

df = pd.DataFrame(raw_examples, columns=["label_name", "text"])
df["label"] = df["label_name"].map(label2id)

display(df.head())
display(df["label_name"].value_counts().rename("count"))


,label_name,text,label
0,TECHNICAL,Додаток зависає після входу в акаунт.,0
1,TECHNICAL,"Не можу скинути пароль, посилання не відкриває...",0
2,TECHNICAL,На сторінці профілю постійно з'являється помил...,0
3,TECHNICAL,Файл завантажується до 80 відсотків і потім пр...,0
4,TECHNICAL,Після оновлення не працює пошук у кабінеті.,0


label_name
TECHNICAL    8
BILLING      8
DELIVERY     8
Name: count, dtype: int64

## 3. Train/validation split

На малих наборах випадковий split легко ламає баланс класів. Тому для демонстрації беремо по 2 приклади кожного класу у validation, решту залишаємо для train.


In [3]:
shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
validation_idx = shuffled.groupby("label_name", group_keys=False).head(2).index

validation_df = shuffled.loc[validation_idx].reset_index(drop=True)
train_df = shuffled.drop(validation_idx).reset_index(drop=True)

raw_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(validation_df),
})

print(raw_dataset)
print("Train class counts:")
print(train_df["label_name"].value_counts())
print("\nValidation class counts:")
print(validation_df["label_name"].value_counts())


DatasetDict({
    train: Dataset({
        features: ['label_name', 'text', 'label'],
        num_rows: 18
    })
    validation: Dataset({
        features: ['label_name', 'text', 'label'],
        num_rows: 6
    })
})
Train class counts:
label_name
BILLING      6
DELIVERY     6
TECHNICAL    6
Name: count, dtype: int64

Validation class counts:
label_name
BILLING      2
DELIVERY     2
TECHNICAL    2
Name: count, dtype: int64


## 4. Tokenizer

Tokenizer перетворює текст на `input_ids` і `attention_mask`. Для mini-batch використовуємо dynamic padding через `DataCollatorWithPadding`, щоб не падити всі приклади до однакової максимальної довжини наперед.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_dataset = raw_dataset.map(tokenize_batch, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text", "label_name"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

example = raw_dataset["train"][0]
encoded = tokenizer(example["text"], truncation=True, max_length=128)

print(example)
print("\nToken ids:", encoded["input_ids"][:20])
print("Decoded:", tokenizer.decode(encoded["input_ids"]))


## 5. Модель для sequence classification

`AutoModelForSequenceClassification` бере pretrained DistilBERT encoder і додає класифікаційну голову. Під час fine-tuning оновлюються і encoder weights, і нова classification head.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable share:      {trainable_params / total_params:.1%}")


## 6. Метрики та `Trainer`

Для простоти рахуємо accuracy вручну. У production-проєкті додайте macro F1, confusion matrix і окремий test set, який не використовується під час вибору гіперпараметрів.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = float(np.mean(predictions == labels))
    return {"accuracy": accuracy}

def make_training_arguments(**kwargs):
    try:
        return TrainingArguments(eval_strategy="epoch", **kwargs)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **kwargs)

training_args = make_training_arguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=1,
    report_to="none",
    seed=SEED,
)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_signature = inspect.signature(Trainer.__init__)
if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
trainer


## 7. Запуск fine-tuning

Для демонстрації на занятті:

1. перевірте, що модель і tokenizer завантажились;
2. поставте `RUN_TRAINING = True` у першій клітинці налаштувань;
3. запустіть notebook зверху вниз.

На CPU це може зайняти помітний час. На GPU/MPS - значно швидше.


In [ ]:
if RUN_TRAINING:
    train_result = trainer.train()
    print(train_result)
else:
    print("RUN_TRAINING=False: тренування пропущено. Поставте True для повного fine-tuning.")


## 8. Оцінювання

Якщо training пропущено, accuracy не рахуємо: classification head щойно створена і ще не навчена, тому результат буде випадковим.


In [ ]:
if RUN_TRAINING:
    eval_result = trainer.evaluate()
    print(eval_result)
else:
    print("Evaluation пропущено, бо модель ще не fine-tuned.")


## 9. Inference після fine-tuning

Після навчання можна зберегти модель і tokenizer, а потім використовувати їх через `pipeline`. Це той самий принцип, який застосовують для deployment, тільки в production ще додають versioning, monitoring і регулярне переоцінювання якості.


In [ ]:
test_texts = [
    "Не можу оплатити підписку корпоративною карткою.",
    "Після входу в кабінет сторінка зависає.",
    "Потрібно змінити адресу доставки замовлення.",
]

if RUN_TRAINING:
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    classifier = pipeline(
        "text-classification",
        model=str(OUTPUT_DIR),
        tokenizer=str(OUTPUT_DIR),
        device=0 if torch.cuda.is_available() else -1,
    )
    predictions = classifier(test_texts)
    display(pd.DataFrame({"text": test_texts, "prediction": predictions}))
else:
    print("Inference пропущено: спочатку запустіть fine-tuning.")


## 10. Що саме ми fine-tuned?

У цьому прикладі fine-tuning означає:

- беремо pretrained DistilBERT, який уже має мовні представлення;
- додаємо classification head для наших 3 класів;
- показуємо моделі приклади `text -> label`;
- оновлюємо ваги так, щоб logits потрібного класу ставали більшими.

Це відрізняється від RAG: fine-tuning змінює поведінку моделі, а RAG додає зовнішні знання в prompt/context без зміни ваг.
